In [1]:
# load libraries
import re
import pandas as pd
import numpy as np
import sys
import os
import ast
os.environ["TOKENIZERS_PARALLELISM"] = "false"
from transformers import AutoTokenizer, AutoModelForTokenClassification
import torch
from pathlib import Path

project_root = Path.cwd() / "../"
sys.path.append(str(project_root))

from torch.utils.data import Dataset, DataLoader
import json
from tqdm import tqdm
from utils.evaluation import labels_to_wordlevel_tags

### **Add Days Till Election Variable**

In [109]:
# add days till election variable
df = pd.read_csv(
    "../01_data/empirical_analysis/main_speech_datasets/researchperiod_sentencelevel_socialgroups_stances.csv",
    parse_dates=["date", "birth_date"],
    dtype={
        "agenda": "string",
        "text": "string",
        "sentence": "string",
        "speech_id": "int64",
        "speechnumber": "int64",
        "speaker": "string",
        "party": "category",
        "chair": "boolean",
        "age": "int64",
        "gender": "category",
        "vulnerability": "float64",
        "backbencher": "boolean",
        "constituency_name": "string",
        "ons_id": "string",
        "under_30": "float64",
        "over_65": "float64",
    },
    converters={
        "sg_spans": ast.literal_eval,
        "stances": ast.literal_eval,
    }
)

print(f"Number of columns before: {len(df.columns)}")
print(f"Number of rows before: {len(df)}")

election_dates = pd.to_datetime(["2010-05-06", "2015-05-07", "2017-06-08", "2019-12-12"])

idx = election_dates.searchsorted(df["date"], side="right")
df["days_until_election"] = [
    (election_dates[i] - d).days if i < len(election_dates) else None
    for d, i in zip(df["date"], idx)
]


col_order = ["date", "days_until_election", "agenda", "text", "sentence", "sg_spans", "stances", "speech_id", "speechnumber", "speaker", "party", "chair", "age", "gender", "birth_date", "vulnerability", "backbencher", "constituency_name", "ons_id", "under_30", "over_65"]
df = df.loc[:, col_order]

print(f"Number of columns after: {len(df.columns)}")
print(f"Number of rows after: {len(df)}")

df.head()

Number of columns before: 19
Number of rows before: 5075040
Number of columns after: 20
Number of rows after: 5075040


,date,days_until_election,agenda,text,sentence,sg_spans,speech_id,speechnumber,speaker,party,chair,age,gender,birth_date,vulnerability,backbencher,constituency_name,ons_id,under_30,over_65
0,2010-05-18,1815,Election of Speaker,"On a point of order, Sir Peter. May I ask a pr...","On a point of order, Sir Peter.",[],0,3,Jim Sheridan,Lab,False,57,male,1952-11-24,34.960075,True,Paisley and Renfrewshire North,S14000052,34.811785,16.795834
1,2010-05-18,1815,Election of Speaker,"On a point of order, Sir Peter. May I ask a pr...",May I ask a procedural question?,[],0,3,Jim Sheridan,Lab,False,57,male,1952-11-24,34.960075,True,Paisley and Renfrewshire North,S14000052,34.811785,16.795834
2,2010-05-18,1815,Election of Speaker,"On a point of order, Sir Peter. May I ask a pr...",This is an extremely important time for this H...,[],0,3,Jim Sheridan,Lab,False,57,male,1952-11-24,34.960075,True,Paisley and Renfrewshire North,S14000052,34.811785,16.795834
3,2010-05-18,1815,Election of Speaker,"On a point of order, Sir Peter. May I ask a pr...",We are in the process of electing a Speaker wi...,[],0,3,Jim Sheridan,Lab,False,57,male,1952-11-24,34.960075,True,Paisley and Renfrewshire North,S14000052,34.811785,16.795834
4,2010-05-18,1815,Election of Speaker,"On a point of order, Sir Peter. May I ask a pr...",May I therefore ask what safeguards are in pla...,[],0,3,Jim Sheridan,Lab,False,57,male,1952-11-24,34.960075,True,Paisley and Renfrewshire North,S14000052,34.811785,16.795834


In [113]:
df.to_csv("../01_data/empirical_analysis/main_speech_datasets/researchperiod_sentencelevel_socialgroups.csv", index=False)

In [3]:
# load training set
with open("../01_data/classification/training_validation_sets/ner/training_set.json", "r") as f:
    training_set = json.load(f)

# load the main speech dataset
main_speech_df = pd.read_csv(
    "../01_data/empirical_analysis/main_speech_datasets/researchperiod_sentencelevel_socialgroups_stances.csv",
    parse_dates=["date", "birth_date"],
    dtype={
        "agenda": "string",
        "sentence": "string",
        "speech_id": "int64",
        "speechnumber": "int64",
        "speaker": "string",
        "party": "category",
        "chair": "boolean",
        "age": "int64",
        "gender": "category",
        "vulnerability": "float64",
        "backbencher": "boolean",
        "constituency_name": "string",
        "ons_id": "string",
        "under_30": "float64",
        "over_65": "float64",
    },
    converters={
        "sg_spans": ast.literal_eval,
        "stances": ast.literal_eval,
    }
)
# initialize tag dictionary
tag_dict = {"O"}

# loop through all sentences
for task in training_set:
    if task["annotations"]:
        for annotation in task["annotations"]:
            label = annotation["tag"][0:2]
            tag_dict.add(f"B-{label}")
            tag_dict.add(f"I-{label}")

# sort the tag dictionary
tag_list = sorted(tag_dict)

# dictionaries that convert from id to tag and vice versa
tag_to_id = {tag: i for i, tag in enumerate(tag_list)}
id_to_tag = {id: label for label, id in tag_to_id.items()}

tokenizer = AutoTokenizer.from_pretrained("roberta-base")
device = 'mps' if torch.backends.mps.is_available() else 'cpu'
model = AutoModelForTokenClassification.from_pretrained(
     "roberta-base",
     num_labels=len(tag_to_id),
     id2label=id_to_tag,
     label2id=tag_to_id
     ).to(device)
model.load_state_dict(torch.load(f="../04_classification_evaluation/final_model_application/final_models/social_group_detection_model.pt", map_location=torch.device("mps")))

Some weights of RobertaForTokenClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


<All keys matched successfully>

In [9]:
# reduce df to all speeches given during parliamentary question debates
question_df = main_speech_df[main_speech_df["agenda"].str.contains("question", case=False, na=False)].copy()

# get all detected social group mentions as a df
mentions_df = question_df.loc[:, ["sg_spans"]].explode("sg_spans").dropna().reset_index(names="row_id")


test_df = main_speech_df.loc[[11620]]
#test_df = main_speech_df.iloc[0:100, :]


class InferenceTokenDataset(Dataset):
    def __init__(self, df, tokenizer, max_len):
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        text = row["sentence"]

        encoding = self.tokenizer(text, return_offsets_mapping=True, truncation=True, max_length=self.max_len, padding="max_length", return_tensors="pt")

        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "offset_mapping": encoding["offset_mapping"].squeeze(0),
            "word_ids": encoding.word_ids()
        }
    
def collate_bert_ner(batch):
    return {
        "input_ids": torch.stack([item["input_ids"] for item in batch]),
        "attention_mask": torch.stack([item["attention_mask"] for item in batch]),
        "offset_mapping": torch.stack([item["offset_mapping"] for item in batch]),
        "word_ids": [item["word_ids"] for item in batch]
    }

# turn the entire dataset to class object
sentence_level_dataset = InferenceTokenDataset(test_df, tokenizer, 128)
sentence_level_dataloader = DataLoader(sentence_level_dataset, batch_size=128, shuffle=False, collate_fn=collate_bert_ner)

In [10]:
def bio_tags_to_spans(word_tags, word_offsets):
    spans = []
    start = None

    for i, tag in enumerate(word_tags):
        if tag == "B":
            if start is not None:
                spans.append((start, word_offsets[i - 1][1]))
            start = word_offsets[i][0]

        elif tag == "I":
            continue

        else:
            if start is not None:
                spans.append((start, word_offsets[i - 1][1]))
                start = None

    if start is not None:
        spans.append((start, word_offsets[len(word_tags) - 1][1]))

    return spans

model.eval()

all_results = []
   
with torch.no_grad():
      
      progress_bar = tqdm(sentence_level_dataloader, desc="Inference")
      for batch in progress_bar:
        input_ids = batch["input_ids"].to(device)
        attention_masks = batch["attention_mask"].to(device)
        offset_mappings = batch["offset_mapping"]
        word_ids = batch["word_ids"]

        outputs = model(input_ids=input_ids, attention_mask=attention_masks)
        logits = outputs.logits
        predictions = torch.argmax(logits, dim=2)

        for sent_input_ids, preds, sent_word_ids, offsets in zip(input_ids, predictions, word_ids, offset_mappings):
            pred_np_array = preds.cpu().numpy()
            tokens = tokenizer.convert_ids_to_tokens(sent_input_ids.cpu().tolist())
            word_tags, _ = labels_to_wordlevel_tags(pred_np_array, id_to_tag, sent_word_ids)

            word_offsets = []
            prev_word_id = None
            start_offset = None

            for off, wid in zip(offsets, sent_word_ids):
                if wid is None:
                    continue

                if wid != prev_word_id:
                    # finalize previous word
                    if start_offset is not None:
                        word_offsets.append((start_offset, prev_end_offset))

                    # start new word
                    start_offset = off[0]
                    prev_word_id = wid

                # always update end offset
                prev_end_offset = off[1]

            # finalize last word
            if start_offset is not None:
                word_offsets.append((start_offset, prev_end_offset))

            spans = bio_tags_to_spans(word_tags, word_offsets)
            all_results.append(spans)

all_text_spans = []

for spans, text in zip(all_results, test_df["sentence"]):
    sent_spans = [text[start:end] for start, end in spans]
    all_text_spans.append(sent_spans)

Inference: 100%|██████████| 1/1 [00:00<00:00,  8.58it/s]


In [12]:
test_df["sentence"].iloc[0]

'Will he have a word with those hypocrites, and every time they talk about the Post Office, remind them of that?'

In [17]:
for w, p in zip(sent_word_ids, pred_np_array):
    print(w, p)

None 2
0 2
1 2
2 2
3 2
4 2
5 2
6 0
7 1
7 2
8 2
9 2
10 2
11 2
12 2
13 2
14 2
15 2
16 2
17 2
18 2
19 2
20 2
21 2
22 2
23 2
None 2
None 2
None 2
None 2
None 2
None 2
None 2
None 2
None 2
None 2
None 2
None 2
None 2
None 2
None 2
None 2
None 2
None 2
None 2
None 2
None 2
None 2
None 2
None 2
None 2
None 2
None 2
None 2
None 2
None 2
None 2
None 2
None 2
None 2
None 2
None 2
None 2
None 2
None 2
None 2
None 2
None 2
None 2
None 2
None 2
None 2
None 2
None 2
None 2
None 2
None 2
None 2
None 2
None 2
None 2
None 2
None 2
None 2
None 2
None 2
None 2
None 2
None 2
None 2
None 2
None 2
None 2
None 2
None 2
None 2
None 2
None 2
None 2
None 2
None 2
None 2
None 2
None 2
None 2
None 2
None 2
None 2
None 2
None 2
None 2
None 2
None 2
None 2
None 2
None 2
None 2
None 2
None 2
None 2
None 2
None 2
None 2
None 2
None 2
None 2
None 2
None 2
